In [ ]:
# ============================================================
# 🥔 PL-LDGAN IMAGE GENERATION (MATCHED TO TRAINED MODEL)
# ============================================================
import torch, os, zipfile
from torch import nn
from torchvision.utils import save_image

# ---------------- CONFIG ----------------
LATENT_DIM = 256           
IMG_SIZE = 256
NUM_IMAGES = 5000
TIMESTEPS = 500

DATA_PATH = "/content/drive/MyDrive/POTATODATASET/Early_blight"
CKPT_PATH = "/content/drive/MyDrive/POTATODATASET/LDM_GAN_LIGHT/checkpoints"
OUTPUT_DIR = "/content/drive/MyDrive/POTATODATASET/LDM_GAN_LIGHT/GeneratedEB"

LATEST_EPOCH = 1000        # 🔴 change if needed

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------- DECODER (EXACT MATCH) ----------------
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(LATENT_DIM, 256 * 16 * 16)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.ReLU(),  # 16 → 32
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.ReLU(),   # 32 → 64
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),    # 64 → 128
            nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Tanh()      # 128 → 256
        )

    def forward(self, z):
        z = self.fc(z).view(-1, 256, 16, 16)
        return self.net(z)

# ---------------- LATENT DIFFUSION (MATCH) ----------------
class LatentDiffusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(TIMESTEPS, LATENT_DIM)
        self.net = nn.Sequential(
            nn.Linear(LATENT_DIM, LATENT_DIM),
            nn.ReLU(),
            nn.Linear(LATENT_DIM, LATENT_DIM)
        )

    def forward(self, z, t):
        return self.net(z + self.embed(t))

# ---------------- LOAD MODELS ----------------
decoder = Decoder().to(device)
ldm = LatentDiffusion().to(device)

decoder.load_state_dict(
    torch.load(f"{CKPT_PATH}/decoder_{LATEST_EPOCH}.pth", map_location=device)
)
ldm.load_state_dict(
    torch.load(f"{CKPT_PATH}/ldm_{LATEST_EPOCH}.pth", map_location=device)
)

decoder.eval()
ldm.eval()

print(f"✅ Loaded decoder & LDM @ epoch {LATEST_EPOCH}")

# ---------------- DIFFUSION SCHEDULE ----------------
betas = torch.linspace(1e-4, 0.02, TIMESTEPS).to(device)
alphas = 1 - betas
alpha_hat = torch.cumprod(alphas, dim=0)

# ---------------- SAMPLING ----------------
@torch.no_grad()
def sample_latents(n):
    z = torch.randn(n, LATENT_DIM, device=device)

    for t in reversed(range(TIMESTEPS)):
        t_batch = torch.full((n,), t, device=device, dtype=torch.long)
        eps = ldm(z, t_batch)

        a = alphas[t]
        ah = alpha_hat[t]

        z = (1 / torch.sqrt(a)) * (z - (1 - a) / torch.sqrt(1 - ah) * eps)

        if t > 0:
            z += torch.sqrt(betas[t]) * torch.randn_like(z)

    return z

# ---------------- GENERATION ----------------
print("🚀 Generating images...")
count = 0
BATCH_GEN = 32

while count < NUM_IMAGES:
    bs = min(BATCH_GEN, NUM_IMAGES - count)
    z = sample_latents(bs)
    imgs = decoder(z)
    imgs = (imgs + 1) / 2

    for i in range(bs):
        save_image(imgs[i], f"{OUTPUT_DIR}/potatoEB_{count:05}.png")
        count += 1

    print(f"✔ {count}/{NUM_IMAGES}")

print("🎉 Generation complete")

# ---------------- ZIP ----------------
zip_path = OUTPUT_DIR + ".zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(OUTPUT_DIR):
        z.write(os.path.join(OUTPUT_DIR, f), arcname=f)

print(f"📦 ZIP saved at: {zip_path}")


In [ ]:
# ============================================================
# 🥔 PL-LDGAN IMAGE GENERATION (MATCHED TO TRAINED MODEL)
# ============================================================

import torch, os, zipfile
from torch import nn
from torchvision.utils import save_image

# ---------------- CONFIG ----------------
LATENT_DIM = 256          
IMG_SIZE = 256
NUM_IMAGES = 5000
TIMESTEPS = 500

DATA_PATH = "/content/drive/MyDrive/POTATODATASET/Late_blight"
CKPT_PATH = "/content/drive/MyDrive/POTATODATASET/LDM_GAN_LIGHT/checkpoints"
OUTPUT_DIR = "/content/drive/MyDrive/POTATODATASET/LDM_GAN_LIGHT/GeneratedLB"

LATEST_EPOCH = 1000        # 🔴 change if needed

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------- DECODER (EXACT MATCH) ----------------
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(LATENT_DIM, 256 * 16 * 16)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.ReLU(),  # 16 → 32
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.ReLU(),   # 32 → 64
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),    # 64 → 128
            nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Tanh()      # 128 → 256
        )

    def forward(self, z):
        z = self.fc(z).view(-1, 256, 16, 16)
        return self.net(z)

# ---------------- LATENT DIFFUSION (MATCH) ----------------
class LatentDiffusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(TIMESTEPS, LATENT_DIM)
        self.net = nn.Sequential(
            nn.Linear(LATENT_DIM, LATENT_DIM),
            nn.ReLU(),
            nn.Linear(LATENT_DIM, LATENT_DIM)
        )

    def forward(self, z, t):
        return self.net(z + self.embed(t))

# ---------------- LOAD MODELS ----------------
decoder = Decoder().to(device)
ldm = LatentDiffusion().to(device)

decoder.load_state_dict(
    torch.load(f"{CKPT_PATH}/decoder_{LATEST_EPOCH}.pth", map_location=device)
)
ldm.load_state_dict(
    torch.load(f"{CKPT_PATH}/ldm_{LATEST_EPOCH}.pth", map_location=device)
)

decoder.eval()
ldm.eval()

print(f"✅ Loaded decoder & LDM @ epoch {LATEST_EPOCH}")

# ---------------- DIFFUSION SCHEDULE ----------------
betas = torch.linspace(1e-4, 0.02, TIMESTEPS).to(device)
alphas = 1 - betas
alpha_hat = torch.cumprod(alphas, dim=0)

# ---------------- SAMPLING ----------------
@torch.no_grad()
def sample_latents(n):
    z = torch.randn(n, LATENT_DIM, device=device)

    for t in reversed(range(TIMESTEPS)):
        t_batch = torch.full((n,), t, device=device, dtype=torch.long)
        eps = ldm(z, t_batch)

        a = alphas[t]
        ah = alpha_hat[t]

        z = (1 / torch.sqrt(a)) * (z - (1 - a) / torch.sqrt(1 - ah) * eps)

        if t > 0:
            z += torch.sqrt(betas[t]) * torch.randn_like(z)

    return z

# ---------------- GENERATION ----------------
print("🚀 Generating images...")
count = 0
BATCH_GEN = 32

while count < NUM_IMAGES:
    bs = min(BATCH_GEN, NUM_IMAGES - count)
    z = sample_latents(bs)
    imgs = decoder(z)
    imgs = (imgs + 1) / 2

    for i in range(bs):
        save_image(imgs[i], f"{OUTPUT_DIR}/potatoEB_{count:05}.png")
        count += 1

    print(f"✔ {count}/{NUM_IMAGES}")

print("🎉 Generation complete")

# ---------------- ZIP ----------------
zip_path = OUTPUT_DIR + ".zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(OUTPUT_DIR):
        z.write(os.path.join(OUTPUT_DIR, f), arcname=f)

print(f"📦 ZIP saved at: {zip_path}")


In [ ]:
# ============================================================
# 🥔 PL-LDGAN IMAGE GENERATION (MATCHED TO TRAINED MODEL)
# ============================================================

import torch, os, zipfile
from torch import nn
from torchvision.utils import save_image

# ---------------- CONFIG ----------------
LATENT_DIM = 256          
IMG_SIZE = 256
NUM_IMAGES = 5000
TIMESTEPS = 500

DATA_PATH = "/content/drive/MyDrive/POTATODATASET/Healthy"
CKPT_PATH = "/content/drive/MyDrive/POTATODATASET/LDM_GAN_LIGHT/checkpoints"
OUTPUT_DIR = "/content/drive/MyDrive/POTATODATASET/LDM_GAN_LIGHT/GeneratedHealthy"

LATEST_EPOCH = 1000        # 🔴 change if needed

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------- DECODER (EXACT MATCH) ----------------
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(LATENT_DIM, 256 * 16 * 16)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.ReLU(),  # 16 → 32
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.ReLU(),   # 32 → 64
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),    # 64 → 128
            nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Tanh()      # 128 → 256
        )

    def forward(self, z):
        z = self.fc(z).view(-1, 256, 16, 16)
        return self.net(z)

# ---------------- LATENT DIFFUSION (MATCH) ----------------
class LatentDiffusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(TIMESTEPS, LATENT_DIM)
        self.net = nn.Sequential(
            nn.Linear(LATENT_DIM, LATENT_DIM),
            nn.ReLU(),
            nn.Linear(LATENT_DIM, LATENT_DIM)
        )

    def forward(self, z, t):
        return self.net(z + self.embed(t))

# ---------------- LOAD MODELS ----------------
decoder = Decoder().to(device)
ldm = LatentDiffusion().to(device)

decoder.load_state_dict(
    torch.load(f"{CKPT_PATH}/decoder_{LATEST_EPOCH}.pth", map_location=device)
)
ldm.load_state_dict(
    torch.load(f"{CKPT_PATH}/ldm_{LATEST_EPOCH}.pth", map_location=device)
)

decoder.eval()
ldm.eval()

print(f"✅ Loaded decoder & LDM @ epoch {LATEST_EPOCH}")

# ---------------- DIFFUSION SCHEDULE ----------------
betas = torch.linspace(1e-4, 0.02, TIMESTEPS).to(device)
alphas = 1 - betas
alpha_hat = torch.cumprod(alphas, dim=0)

# ---------------- SAMPLING ----------------
@torch.no_grad()
def sample_latents(n):
    z = torch.randn(n, LATENT_DIM, device=device)

    for t in reversed(range(TIMESTEPS)):
        t_batch = torch.full((n,), t, device=device, dtype=torch.long)
        eps = ldm(z, t_batch)

        a = alphas[t]
        ah = alpha_hat[t]

        z = (1 / torch.sqrt(a)) * (z - (1 - a) / torch.sqrt(1 - ah) * eps)

        if t > 0:
            z += torch.sqrt(betas[t]) * torch.randn_like(z)

    return z

# ---------------- GENERATION ----------------
print("🚀 Generating images...")
count = 0
BATCH_GEN = 32

while count < NUM_IMAGES:
    bs = min(BATCH_GEN, NUM_IMAGES - count)
    z = sample_latents(bs)
    imgs = decoder(z)
    imgs = (imgs + 1) / 2

    for i in range(bs):
        save_image(imgs[i], f"{OUTPUT_DIR}/potatoEB_{count:05}.png")
        count += 1

    print(f"✔ {count}/{NUM_IMAGES}")

print("🎉 Generation complete")

# ---------------- ZIP ----------------
zip_path = OUTPUT_DIR + ".zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(OUTPUT_DIR):
        z.write(os.path.join(OUTPUT_DIR, f), arcname=f)

print(f"📦 ZIP saved at: {zip_path}")
